In [1]:
#Import pandas library to work with datasets.
import pandas as pd

In [2]:
#Install openpyxl to enable reading Excels.
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
sos_total_og = pd.read_excel("../2020_total_gen_election.xlsx")
sos_gender_og = pd.read_excel("../2020_gender_gen_election.xlsx")
sos_age_og = pd.read_excel("../2020_age_gen_election.xlsx")
sos_race_og = pd.read_excel("../2020_race_gen_election.xlsx")
rdh_og = pd.read_csv("../data_storage/raw_data/voter_data/AL_l2_2024_gen_stats_2020county.csv")
alvr_og = pd.read_excel("../data_storage/raw_data/voter_data/2020_votereg_data_AlSoS.xlsx", sheet_name="October", header=None, engine="openpyxl")

FileNotFoundError: [Errno 2] No such file or directory: '../2020_total_gen_election.xlsx'

In [113]:
sos_total = sos_total_og.copy()
sos_gender = sos_gender_og.copy()
sos_age = sos_age_og.copy()
sos_race = sos_race_og.copy()
rdh = rdh_og.copy()
alvr = alvr_og.copy()

In [114]:
#Standardize county name to title case across all SoS files
sos_total["county"] = sos_total["county"].str.strip().str.title()
sos_gender["county"] = sos_gender["county"].str.strip().str.title()
sos_age["county"] = sos_age["county"].str.strip().str.title()
sos_race["county"] = sos_race["county"].str.strip().str.title()

#RDH uses different column names so rename and standardize
rdh = rdh.rename(columns={"countyname": "county", "countyfips": "fips_code"})
rdh["county"] = rdh["county"].str.strip().str.title()

#ALVR has a broken two-row header so manually set clean column names
alvr.columns = ['county','Total_AI','Active_Asian','Active_AmInd','Active_Black','Active_Fed','Active_Hispanic','Active_Korean','Active_White','Active_Other','Active_NotId','Active_Total','Inactive_Asian','Inactive_AmInd','Inactive_Black','Inactive_Fed','Inactive_Hispanic','Inactive_Korean','Inactive_White','Inactive_Other','Inactive_NotId','Inactive_Total']

#Drops rows 0 and 1 which contained the broken header labels then resets index
alvr = alvr.iloc[2:].reset_index(drop=True)
alvr["county"] = alvr["county"].astype(str).str.strip().str.title()

#Fill NaN in ALVR columns with 0 to handle missing values like Wilcox Inactive Hispanic
alvr_numeric_columns = ['Active_Asian','Active_AmInd','Active_Black','Active_Fed','Active_Hispanic',
    'Active_Korean','Active_White','Active_Other','Active_NotId','Active_Total',
    'Inactive_Asian','Inactive_AmInd','Inactive_Black','Inactive_Fed','Inactive_Hispanic',
    'Inactive_Korean','Inactive_White','Inactive_Other','Inactive_NotId','Inactive_Total']
alvr[alvr_numeric_columns] = alvr[alvr_numeric_columns].fillna(0)

/var/folders/_7/6hr00gjd7rgcmfz744n7rc2h0000gn/T/ipykernel_67419/255743878.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  alvr[alvr_numeric_columns] = alvr[alvr_numeric_columns].fillna(0)


In [115]:
#Filters the 17 counties for Alabama BB
BLACK_BELT_COUNTIES = ["Sumter", "Choctaw", "Greene", "Hale", "Marengo", "Perry", "Dallas", "Wilcox", "Lowndes", "Butler", "Crenshaw", "Montgomery", "Pike", "Bullock", "Macon", "Barbour", "Russell"]

sos_total = sos_total[sos_total["county"].isin(BLACK_BELT_COUNTIES)].copy()
sos_gender = sos_gender[sos_gender["county"].isin(BLACK_BELT_COUNTIES)].copy()
sos_age = sos_age[sos_age["county"].isin(BLACK_BELT_COUNTIES)].copy()
sos_race = sos_race[sos_race["county"].isin(BLACK_BELT_COUNTIES)].copy()
rdh_bb_filtered = rdh[rdh["county"].isin(BLACK_BELT_COUNTIES)].copy()
alvr_bb_filtered = alvr[alvr["county"].isin(BLACK_BELT_COUNTIES)].copy()

In [116]:
#Slim each SoS breakdown file before merging to avoid duplicate total_ballots columns
gender_slim = sos_gender[["county", "total_female", "total_male", "total_unidentified"]]
age_slim = sos_age[["county", "age_18_29", "age_30_39", "age_40_49", "age_50_59", "age_60_69", "age_70_79", "age_80_89", "age_90_99", "age_100_plus"]]
race_slim = sos_race[["county", "black_ballots", "white_ballots", "hispanic_ballots"]]

In [117]:
#Inner join to merge all four SoS files into one participation dataframe, add FIPS code from RDH and total registered by race from ALVR
participation = sos_total.merge(gender_slim, on="county").merge(age_slim, on="county").merge(race_slim, on="county")
participation = participation.merge(rdh_bb_filtered[["county", "fips_code"]], on="county")
participation = participation.merge(alvr_bb_filtered[["county", "Active_Black", "Active_White", "Active_Hispanic", "Inactive_Black", "Inactive_White", "Inactive_Hispanic"]], on="county")

In [118]:
#Combine age buckets 80-89, 90-99, 100+ into single 80+ column
participation["ballots_age_80_plus"] = participation["age_80_89"] + participation["age_90_99"] + participation["age_100_plus"]

In [119]:
#Calculate total registered by race from ALVR (active + inactive) to use as participation rate denominator
participation["total_registered_black"] = participation["Active_Black"].astype(float) + participation["Inactive_Black"].astype(float)
participation["total_registered_white"] = participation["Active_White"].astype(float) + participation["Inactive_White"].astype(float)
participation["total_registered_latinx"] = participation["Active_Hispanic"].astype(float) + participation["Inactive_Hispanic"].astype(float)

In [120]:
#Calculate race-based participation rates using total registered voters (active + inactive) from ALVR October snapshot
#This is consistent with how overall turnout_rate is calculated — ballots divided by total registered
#Force float conversion on both sides so rounding applies correctly since ALVR columns may be object type
participation["turnout_rate_black"] = (participation["black_ballots"].astype(float) / (participation["Active_Black"].astype(float) + participation["Inactive_Black"].astype(float))).round(4)
participation["turnout_rate_white"] = (participation["white_ballots"].astype(float) / (participation["Active_White"].astype(float) + participation["Inactive_White"].astype(float))).round(4)
participation["turnout_rate_latinx"] = (participation["hispanic_ballots"].astype(float) / (participation["Active_Hispanic"].astype(float) + participation["Inactive_Hispanic"].astype(float))).round(4)

In [121]:
#Rename only columns that need to change per naming convention
participation = participation.rename(columns={
    #Geographic
    "county": "county_name",

    #Overall totals
    "total_ballots": "total_ballots_cast",
    "turnout_percent": "turnout_rate",
    "absentee_percent": "absentee_rate",
    
    #Gender columns
    "total_female": "ballots_female",
    "total_male": "ballots_male",
    "total_unidentified": "ballots_not_identified_gender",

    #Age columns
    "age_18_29": "ballots_age_18_29",
    "age_30_39": "ballots_age_30_39",
    "age_40_49": "ballots_age_40_49",
    "age_50_59": "ballots_age_50_59",
    "age_60_69": "ballots_age_60_69",
    "age_70_79": "ballots_age_70_79",

    #Race (rename to ballots prefix for consistency and hispanic to latinx per SPLC language)
    "black_ballots": "ballots_black",
    "white_ballots": "ballots_white",
    "hispanic_ballots": "ballots_latinx"
})

In [122]:
#Keep only final output columns in the order SPLC will read them
final_columns = [
    #Geographic identifiers
    "county_name", "fips_code",

    #Registration and overall turnout
    "registered_voters", "total_ballots_cast", "absentee_ballots", "turnout_rate", "absentee_rate",

    #Gender breakdown
    "ballots_female", "ballots_male", "ballots_not_identified_gender",

    #Age breakdown
    "ballots_age_18_29", "ballots_age_30_39", "ballots_age_40_49", "ballots_age_50_59",
    "ballots_age_60_69", "ballots_age_70_79", "ballots_age_80_plus",

    #Race breakdown
    "ballots_black", "ballots_white", "ballots_latinx",

    #Race based turnout rates calculated using active + inactive registered from ALVR
    "turnout_rate_black", "turnout_rate_white", "turnout_rate_latinx"
]

participation = participation[final_columns].sort_values("county_name").reset_index(drop=True)

In [123]:
#Display final cleaned participation dataset
participation

#Export to cleaned data folder
participation.to_csv("../data_storage/clean_data/voter_turnout_2024_general.csv", index=False)